# SER - Experiment 05: BOTH leakage mechanisms combined

Experiments 03 and 04 measured each mechanism in isolation:

| Pipeline | Rows | Accuracy | Delta |
|---|---|---|---|
| `base` - split then augment (correct) | 12,162 | 0.5795 | - |
| `base_leaky` - duplicate mirrors | 16,402 | 0.7266 | +14.71 |
| `base_augleak` - augment then split | 36,486 | 0.8223 | +24.27 |
| `base_bothleak` - **both** (this) | 49,206 | ? | ? |
| *base paper reported* | *12,162* | *0.9491* | *+36.96* |

Neither mechanism alone reaches 94.91%, but they are independent and should
compound. This notebook applies both: the duplicated mirrors **and**
augment-before-split.

If the result lands in the 0.90-0.95 band, the base paper's headline is fully
accounted for by two common pipeline errors - and the honest number for this
corpus is 0.5795.

**Attach:** the four corpora only.
**Accelerator: GPU.** Roughly 35 min extraction plus 25 min training.

In [ ]:
import glob
import json
import os
import shutil
import sys
import time

import numpy as np
import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Set Settings -> Accelerator -> GPU before running."

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
# BOTH MECHANISMS. Symlink each dataset ROOT (not the canonical subdirectory)
# so the scan sees both copies of RAVDESS and TESS - then augment before
# splitting in the cell below.
DATA_ROOT = "/kaggle/working/ser/data_bothleak"

CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    canon = find_canonical(target)
    assert canon is not None, f"MISSING INPUT for {name}: no '{target}' found"
    src = os.path.dirname(canon)          # PARENT -> duplicated tree
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(dst, "**", "*.wav"), recursive=True))
    print(f"{name:9s} {n:6d} wav   <- {src}")

In [ ]:
import config
import data_loader

data_loader._CORPORA = [
    ("RAVDESS", os.path.join(DATA_ROOT, "RAVDESS"), data_loader._parse_ravdess),
    ("TESS",    os.path.join(DATA_ROOT, "TESS"),    data_loader._parse_tess),
    ("SAVEE",   os.path.join(DATA_ROOT, "SAVEE"),   data_loader._parse_savee),
    ("CREMA-D", os.path.join(DATA_ROOT, "CREMA-D"), data_loader._parse_cremad),
]

config.CACHE_DIR = "/kaggle/working/features_cache_bothleak"
config.RUNS_DIR = "/kaggle/working/runs"
os.makedirs(config.CACHE_DIR, exist_ok=True)
os.makedirs(config.RUNS_DIR, exist_ok=True)

# MECHANISM 1: duplicate mirrors (strict=False disables the guard).
meta = data_loader.build_metadata(strict=False)
assert len(meta) > 12162, "symlinks are pointing at canonical subdirs"
print()
print(f"utterance rows (with duplicates): {len(meta)}")

# MECHANISM 2: augment every row BEFORE splitting.
N_COPIES = 2
items = []
for i, r in enumerate(meta.itertuples()):
    items.append({"path": r.path, "emotion": r.emotion, "augment": False,
                  "emotion_aware": False, "seed": 0, "src": r.path})
    for k in range(N_COPIES):
        items.append({"path": r.path, "emotion": r.emotion, "augment": True,
                      "emotion_aware": False,
                      "seed": int(config.RANDOM_SEED + 1 + i * N_COPIES + k),
                      "src": r.path})

print(f"rows after augment              : {len(items)}   "
      f"({1 + N_COPIES} per utterance row)")

In [ ]:
from sklearn.model_selection import train_test_split

y_all = [it["emotion"] for it in items]

train_items, test_items = train_test_split(
    items, test_size=config.TEST_FRACTION, stratify=y_all,
    random_state=config.RANDOM_SEED)
train_items, val_items = train_test_split(
    train_items, test_size=config.VAL_FRACTION_OF_TRAINVAL,
    stratify=[it["emotion"] for it in train_items],
    random_state=config.RANDOM_SEED)

print(f"[split] train={len(train_items)}  val={len(val_items)}  "
      f"test={len(test_items)}")

# Contamination measured two ways: by source path (augment-order leakage) and
# by basename (duplicate-mirror leakage, which also matches across copies).
seen_src = {it["src"] for it in train_items} | {it["src"] for it in val_items}
seen_base = {os.path.basename(it["src"]) for it in train_items} | \
            {os.path.basename(it["src"]) for it in val_items}

leak_src = sum(1 for it in test_items if it["src"] in seen_src)
leak_base = sum(1 for it in test_items
                if os.path.basename(it["src"]) in seen_base)

print()
print(f"test rows                          : {len(test_items)}")
print(f"same source path in train/val      : {leak_src} "
      f"({100 * leak_src / len(test_items):.1f}%)")
print(f"same RECORDING in train/val        : {leak_base} "
      f"({100 * leak_base / len(test_items):.1f}%)   <- total contamination")

In [ ]:
from features import build_feature_matrix
from utils import StreamScalers, set_seed

set_seed(config.RANDOM_SEED)

train_feats = build_feature_matrix(train_items, desc="bothleak_train")
val_feats = build_feature_matrix(val_items, desc="bothleak_val")
test_feats = build_feature_matrix(test_items, desc="bothleak_test")

scalers = StreamScalers().fit(train_feats)
x_train = scalers.transform(train_feats)
x_val = scalers.transform(val_feats)
x_test = scalers.transform(test_feats)

y_train = tf.keras.utils.to_categorical(train_feats["y"], config.NUM_CLASSES)
y_val = tf.keras.utils.to_categorical(val_feats["y"], config.NUM_CLASSES)

print("train:", x_train[0].shape, "val:", x_val[0].shape,
      "test:", x_test[0].shape)

In [ ]:
from model import build_model

run_dir = os.path.join(config.RUNS_DIR, "base_bothleak")
os.makedirs(run_dir, exist_ok=True)
ckpt_path = os.path.join(run_dir, "best_model.keras")

# Identical to the `base` configuration: no novelties, plain cross-entropy.
model, _ = build_model(use_afw=False, use_mstc=False)
model.compile(optimizer=tf.keras.optimizers.Adam(config.LEARNING_RATE),
              loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                       mode="min", save_best_only=True,
                                       verbose=0),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=config.REDUCE_LR_FACTOR,
        patience=config.REDUCE_LR_PATIENCE, min_lr=config.MIN_LR, verbose=0),
    tf.keras.callbacks.CSVLogger(os.path.join(run_dir, "training_log.csv")),
]

history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                    epochs=config.EPOCHS, batch_size=config.BATCH_SIZE,
                    callbacks=callbacks, verbose=2)

In [ ]:
from evaluate import evaluate_predictions

# Evaluate the checkpointed best model, matching train.py's behaviour.
if os.path.exists(ckpt_path):
    best = tf.keras.models.load_model(ckpt_path, compile=False)
    model.set_weights(best.get_weights())

y_prob = model.predict(x_test, batch_size=config.BATCH_SIZE, verbose=0)
metrics = evaluate_predictions(test_feats["y"], y_prob, out_dir=run_dir,
                               prefix="test")

In [ ]:
CLEAN_BASE = 0.579531   # split -> augment, de-duplicated
DUP_LEAK = 0.726608     # duplicate mirrors only
AUG_LEAK = 0.822280     # augment -> split only
PAPER = 0.9491          # Chourasia et al. (2026)

both = metrics["accuracy"]

print("=" * 72)
print("  COMPLETE LEAKAGE ACCOUNTING")
print("=" * 72)
print(f"  base          correct pipeline,   12,162 rows : {CLEAN_BASE:.4f}")
print(f"  base_leaky    duplicate mirrors,  16,402 rows : {DUP_LEAK:.4f}"
      f"   ({100 * (DUP_LEAK - CLEAN_BASE):+6.2f} pts)")
print(f"  base_augleak  augment -> split,   36,486 rows : {AUG_LEAK:.4f}"
      f"   ({100 * (AUG_LEAK - CLEAN_BASE):+6.2f} pts)")
print(f"  base_bothleak BOTH mechanisms,    {len(items):,} rows : {both:.4f}"
      f"   ({100 * (both - CLEAN_BASE):+6.2f} pts)")
print("-" * 72)
print(f"  base paper reported                           : {PAPER:.4f}"
      f"   ({100 * (PAPER - CLEAN_BASE):+6.2f} pts)")
print("=" * 72)
print()

if both >= 0.90:
    print("  FULLY ACCOUNTED FOR. Two common pipeline errors reproduce the")
    print("  published accuracy range on this corpus. With both corrected,")
    print(f"  the honest figure is {CLEAN_BASE:.4f}.")
elif both >= AUG_LEAK + 0.03:
    print("  The mechanisms compound as predicted, approaching but not")
    print("  reaching the reported figure.")
else:
    print("  The mechanisms do not compound as strongly as expected.")

json.dump({"clean_base": CLEAN_BASE, "dup_leak": DUP_LEAK,
           "aug_leak": AUG_LEAK, "both_leak": float(both), "paper": PAPER,
           "n_rows": int(len(items)),
           "contamination_pct": float(100 * leak_base / len(test_items))},
          open("/kaggle/working/bothleak_experiment.json", "w"), indent=2)

In [ ]:
for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)

print("output:", sorted(os.listdir("/kaggle/working")))